# 03 — Feature Engineering

Añadimos dos bloques de features derivadas:
- **M1**: Features temporales — rolling mean/std, diferencias, lags, z-score local
- **M2**: Features cruzadas entre sensores — ratio, deltas, gradiente CO2

Estas features mejoran la capacidad discriminativa del modelo respecto a v2.

**Entrada:** `data/interim/02_datos_inyectados.parquet`  
**Salida:** `data/interim/03_datos_features.parquet`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos del paso anterior

In [ ]:
# Cargar el dataset con anomalías inyectadas
df_trabajo = pd.read_parquet(PARQUET_02)
print(f"Dataset cargado: {df_trabajo.shape}")


## 1. Features temporales (v3 — M1)
Para cada sensor calculamos:
- `rmean_30m`, `rstd_30m`: media y std en ventana de 30 minutos
- `rmean_3h`, `rstd_3h`: media y std en ventana de 3 horas
- `diff1`, `diff30`: diferencia con el punto anterior y con 30 minutos antes
- `lag1`, `lag30`: valor 1 y 30 minutos antes
- `zscore_local`: cuántas desviaciones estándar se aleja del rolling_30m

In [ ]:
# ============================================================
# v3 — M1: Features temporales (rolling, diff, lag)
# v3 — M2: Features cruzadas entre sensores
# ============================================================

if 'df_trabajo' in locals():
    sensores_fe = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
                   'XCO2I', 'XHINV', 'XTINV', 'XTS']
    sensores_fe = [c for c in sensores_fe if c in df_trabajo.columns]

    # ── M1: Rolling ──────────────────────────────────────────────
    W_CORTA = 30   # 30 min
    W_LARGA = 180  # 3 h
    for col in sensores_fe:
        s = df_trabajo[col]
        df_trabajo[f'{col}_rmean_30m'] = s.rolling(W_CORTA, min_periods=5).mean()
        df_trabajo[f'{col}_rstd_30m']  = s.rolling(W_CORTA, min_periods=5).std()
        df_trabajo[f'{col}_rmean_3h']  = s.rolling(W_LARGA, min_periods=30).mean()
        df_trabajo[f'{col}_rstd_3h']   = s.rolling(W_LARGA, min_periods=30).std()

    # ── M1: Diff y lag ───────────────────────────────────────────
    for col in sensores_fe:
        df_trabajo[f'{col}_diff1']  = df_trabajo[col].diff(1)
        df_trabajo[f'{col}_diff30'] = df_trabajo[col].diff(30)
        df_trabajo[f'{col}_lag1']   = df_trabajo[col].shift(1)
        df_trabajo[f'{col}_lag30']  = df_trabajo[col].shift(30)

    # ── M1: Z-score local ────────────────────────────────────────
    for col in sensores_fe:
        df_trabajo[f'{col}_zscore_local'] = (
            (df_trabajo[col] - df_trabajo[f'{col}_rmean_30m']) /
            (df_trabajo[f'{col}_rstd_30m'] + 1e-6)
        )

    # ── M2: Features cruzadas ─────────────────────────────────────
    # Ratio radiación interior/exterior
    if 'PRAD' in df_trabajo.columns and 'PRGINT' in df_trabajo.columns:
        df_trabajo['ratio_rad'] = df_trabajo['PRGINT'] / (df_trabajo['PRAD'] + 1.0)

    # Diferencia temperatura interior - exterior
    if 'XTINV' in df_trabajo.columns and 'PTEXT' in df_trabajo.columns:
        df_trabajo['delta_temp'] = df_trabajo['XTINV'] - df_trabajo['PTEXT']

    # Diferencia humedad interior - exterior
    if 'XHINV' in df_trabajo.columns and 'PHEXT' in df_trabajo.columns:
        df_trabajo['delta_hum'] = df_trabajo['XHINV'] - df_trabajo['PHEXT']

    # Gradiente CO2 interior-exterior (respuesta a ventilación)
    if 'XCO2I' in df_trabajo.columns and 'PCO2EXT' in df_trabajo.columns:
        df_trabajo['gradiente_co2'] = df_trabajo['XCO2I'] - df_trabajo['PCO2EXT']

    # Inercia suelo respecto al interior
    if 'XTS' in df_trabajo.columns and 'XTINV' in df_trabajo.columns:
        df_trabajo['delta_temp_suelo'] = df_trabajo['XTS'] - df_trabajo['XTINV']

    nuevas_cols = [c for c in df_trabajo.columns
                   if any(c.endswith(s) for s in
                          ['_rmean_30m','_rstd_30m','_rmean_3h','_rstd_3h',
                           '_diff1','_diff30','_lag1','_lag30','_zscore_local'])
                   or c in ['ratio_rad','delta_temp','delta_hum','gradiente_co2','delta_temp_suelo']]
    print(f"Features añadidas (M1+M2): {len(nuevas_cols)} columnas")
    print(f"Shape total df_trabajo: {df_trabajo.shape}")
else:
    print("Error: df_trabajo no definido.")

## 2. Selección de features
Excluimos columnas de etiquetas, Fecha y UVENT (señal no fiable). El resto son las features de entrada para los modelos.

In [ ]:
# v3: features seleccionadas — excluye UVENT (señal no fiable) e incluye M1+M2
if 'df_trabajo' in locals():
    excluir = ['Fecha', 'etiqueta_deteccion', 'etiqueta_tipo_anomalia',
                'UVENT_cen', 'UVENT_lN']
    columnas_potenciales_features = [c for c in df_trabajo.columns if c not in excluir]

    # Filtrar solo columnas numéricas
    X = df_trabajo[columnas_potenciales_features].select_dtypes(include='number').copy()
    print(f'Features (X): {X.shape[1]} columnas numéricas')
    print(f'Columnas excluidas por no numéricas: '
          f'{[c for c in columnas_potenciales_features if c not in X.columns]}')
    print(X.head())

    # Balance de clases (etiqueta como string, igual que en el resto del pipeline)
    y = df_trabajo['etiqueta_deteccion'].copy()
    print(f'\nBalance de clases:')
    print(y.value_counts())
else:
    print('Error: df_trabajo no definido.')

print(f"Features seleccionadas: {X.shape[1]} columnas")
print(f"Primeras features: {list(X.columns[:10])}")

## 3. Guardar resultado

In [ ]:
# Añadir las features calculadas al dataframe completo y guardar
# (guardamos df_trabajo completo porque contiene etiquetas + features juntas)
df_trabajo.to_parquet(PARQUET_03, index=False)
print(f"Guardado: {PARQUET_03}")
print(f"Shape con features: {df_trabajo.shape}")
